### Load the preprocessed data

In [1]:
import pandas as pd

X_train = pd.read_csv('dataset/X_train.csv')
y_train = pd.read_csv('dataset/y_train.csv').squeeze()
X_test = pd.read_csv('dataset/X_test.csv')
y_test = pd.read_csv('dataset/y_test.csv').squeeze()

X_train_balanced = pd.read_csv('dataset/X_train_balanced.csv')
y_train_balanced = pd.read_csv('dataset/y_train_balanced.csv').squeeze()

print("Raw train:", X_train.shape, "| Balanced train:", X_train_balanced.shape, "| Test:", X_test.shape)

Raw train: (25217, 47) | Balanced train: (39544, 47) | Test: (6305, 47)


### Build a shared results table

In [2]:
from sklearn.model_selection import StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

results = []

def evaluate_and_log(name, model, X_tr, y_tr, X_te, y_te, cv):
    from sklearn.model_selection import cross_val_score
    from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                                  f1_score, roc_auc_score)

    cv_auc = cross_val_score(model, X_tr, y_tr, cv=cv, scoring='roc_auc', n_jobs=-1)
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)
    y_proba = model.predict_proba(X_te)[:, 1]

    row = {
        'model': name,
        'cv_roc_auc_mean': round(cv_auc.mean(), 3),
        'test_accuracy': round(accuracy_score(y_te, y_pred), 3),
        'test_precision': round(precision_score(y_te, y_pred), 3),
        'test_recall': round(recall_score(y_te, y_pred), 3),
        'test_f1': round(f1_score(y_te, y_pred), 3),
        'test_roc_auc': round(roc_auc_score(y_te, y_proba), 3),
    }
    results.append(row)
    print(row)
    return model

### Train the Gradient Boosting baseline (raw training data)

In [4]:
from xgboost import XGBClassifier

gb_baseline = evaluate_and_log('Gradient Boosting (baseline, raw)',
                                 XGBClassifier(eval_metric='logloss', random_state=42),
                                 X_train, y_train, X_test, y_test, cv)

  Using cached xgboost-3.2.0-py3-none-win_amd64.whl.metadata (2.1 kB)
Using cached xgboost-3.2.0-py3-none-win_amd64.whl (101.7 MB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


{'model': 'Gradient Boosting (baseline, raw)', 'cv_roc_auc_mean': np.float64(0.92), 'test_accuracy': 0.911, 'test_precision': 0.898, 'test_recall': 0.663, 'test_f1': 0.763, 'test_roc_auc': 0.922}


### Train on SMOTE-balanced data and compare

In [5]:
gb_balanced = evaluate_and_log('Gradient Boosting (baseline, SMOTE-balanced)',
                                 XGBClassifier(eval_metric='logloss', random_state=42),
                                 X_train_balanced, y_train_balanced, X_test, y_test, cv)

{'model': 'Gradient Boosting (baseline, SMOTE-balanced)', 'cv_roc_auc_mean': np.float64(0.98), 'test_accuracy': 0.909, 'test_precision': 0.877, 'test_recall': 0.675, 'test_f1': 0.763, 'test_roc_auc': 0.924}


### Hyperparameter tuning

In [6]:
from sklearn.model_selection import RandomizedSearchCV

param_dist = {
    'n_estimators': [100, 200, 300, 400],
    'max_depth': [3, 4, 5, 6, 8],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'subsample': [0.7, 0.8, 1.0],
}

search = RandomizedSearchCV(
    XGBClassifier(eval_metric='logloss', random_state=42),
    param_dist, n_iter=20, cv=cv, scoring='roc_auc', n_jobs=-1, random_state=42
)
search.fit(X_train, y_train)

print("Best params:", search.best_params_)
print("Best CV ROC-AUC:", round(search.best_score_, 3))

gb_tuned = evaluate_and_log('Gradient Boosting (tuned)', search.best_estimator_,
                              X_train, y_train, X_test, y_test, cv)

Best params: {'subsample': 1.0, 'n_estimators': 400, 'max_depth': 4, 'learning_rate': 0.1}
Best CV ROC-AUC: 0.928
{'model': 'Gradient Boosting (tuned)', 'cv_roc_auc_mean': np.float64(0.928), 'test_accuracy': 0.912, 'test_precision': 0.922, 'test_recall': 0.647, 'test_f1': 0.76, 'test_roc_auc': 0.929}


### Compare

In [ ]:
pd.DataFrame(results)
# export results to CSV
pd.DataFrame(results).to_csv('dataset/member3_gb_results.csv', index=False)

In [10]:
results

[{'model': 'Gradient Boosting (baseline, raw)',
  'cv_roc_auc_mean': np.float64(0.92),
  'test_accuracy': 0.911,
  'test_precision': 0.898,
  'test_recall': 0.663,
  'test_f1': 0.763,
  'test_roc_auc': 0.922},
 {'model': 'Gradient Boosting (baseline, SMOTE-balanced)',
  'cv_roc_auc_mean': np.float64(0.98),
  'test_accuracy': 0.909,
  'test_precision': 0.877,
  'test_recall': 0.675,
  'test_f1': 0.763,
  'test_roc_auc': 0.924},
 {'model': 'Gradient Boosting (tuned)',
  'cv_roc_auc_mean': np.float64(0.928),
  'test_accuracy': 0.912,
  'test_precision': 0.922,
  'test_recall': 0.647,
  'test_f1': 0.76,
  'test_roc_auc': 0.929}]